# Utils for training Deep Neural Networks

## Setup

In [1]:
import tensorflow as tf

## Dealing with vanishing / exploding gradients

### Initializations

#### Glorot (or Xavier) initialization

Sigmoid activation function (a default option in the past) leads to vanishing / exploding gradients (as sigmoid saturated at 0 and 1 leading to derivatives to be 0). Glorot et al propose initialization teqnique reducing the gradients problem.

By default Keras uses Glorot initialization with a uniform distribution.

#### He

 ReLU activation function helps with unstable gradients, but lead to dying ReLUs problem (as ReLU is saturated at 0). He itialization reducing the problem.

In [2]:
# He normal:
dense = tf.keras.layers.Dense(50, activation="relu",
                              kernel_initializer="he_normal")

In [3]:
# He uniform:
dense = tf.keras.layers.Dense(50, activation="relu",
                              kernel_initializer="he_uniform")

Using *VarianceScaling:*

In [ ]:
# He uniform distr and based on fan_avg (rather than fan_in)

he_avg_init = tf.keras.initializers.VarianceScaling(scale=2., mode="fan_avg", distribution="uniform")

dense = tf.keras.layers.Dense(50, activation="sigmoid",
                              kernel_initializer=he_avg_init)

#### LeCun

### Activation Functions

#### Sigmoid
- saturating

- use Glorot init

#### Tanh
- saturating

#### ReLU
- saturating when *x <= 0* leading to dying ReLUs problem
- not a smooth function -> make gradient descent bounce around the optimum and slow down convergence.

#### Leaky ReLU
- nonsaturating
- $LeakyReLU(z) = max(\alpha z, z)$
- not a smooth function -> make gradient descent bounce around the optimum and slow down convergence.

In [4]:
leaky_relu = tf.keras.layers.LeakyReLU(alpha=0.2)  # defaults to alpha=0.3
dense = tf.keras.layers.Dense(50, activation=leaky_relu,
                              kernel_initializer="he_normal")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [5]:
# LeakyReLU as a separate layer:
model = tf.keras.models.Sequential([
    # [...]  # more layers
    tf.keras.layers.Dense(50, kernel_initializer="he_normal"),  # no activation
    tf.keras.layers.LeakyReLU(alpha=0.2),  # activation as a separate layer
    # [...]  # more layers
])

#### PReLU
- Parametric leaky ReLU: $\alpha$ learnt by backpropagation
- not a smooth function -> make gradient descent bounce around the optimum and slow down convergence.
- Keras: *tf.keras.layers.PReLU*


#### RReLU
- randomized ReLU: $\alpha$ selected randomly
- not implemented in Keras
- not a smooth function -> make gradient descent bounce around the optimum and slow down convergence.

#### ELU
- exponential linear unit
- ELU$_\alpha(z) = \alpha (e^z - 1)$ if $z < 0$, else $z$
- smooth variant of ReLU

In [6]:
dense = tf.keras.layers.Dense(50, activation="elu",
                              kernel_initializer="he_normal")

#### SELU
- scaled ELU
- SELU$(z) = 1.05 \, $ELU$_{1.67}(z)$
- smooth variant of ReLU
- Note: use LeCun init
- if requirements met, network will self-normalize:

By default, the SELU hyperparameters (scale and alpha) are tuned in such a way that the mean output of each neuron remains close to 0, and the standard deviation remains close to 1 (assuming the inputs are standardized with mean 0 and standard deviation 1 too, and other constraints are respected: only MLP, LeCun, no regularization). Using this activation function, even a 1,000 layer deep neural network preserves roughly mean 0 and standard deviation 1 across all layers, avoiding the exploding/vanishing gradients problem:

In [7]:
dense = tf.keras.layers.Dense(50, activation="selu",
                              kernel_initializer="lecun_normal")

#### GELU
- GELU$(z) = z\,\Phi(z)$
- smooth variant of ReLU

#### Swish
- Swish$(z) = z\,\sigma(z)$

#### Mish
- Mish$(z) = z\,\tanh($softplus$(z))$

#### Which to use?

ReLU remains a good default for simple tasks; it’s often just as good as the more sophisticated activation functions, plus it’s very fast to compute, and many libraries and hardware accelerators provide ReLU-specific optimizations.

However, Swish is probably a better default for more complex tasks, and you can even try parameterized Swish with a learnable β parameter for the most complex tasks.

Mish may give you slightly better results, but it requires a bit more compute.

If you care a lot about runtime latency, then you may prefer leaky ReLU or parameterized leaky ReLU for more complex tasks.

For deep MLPs, give SELU a try, but make sure to respect the constraints listed earlier.

If you have spare time and computing power, you can use cross-validation to evaluate other activation functions as well.

### Batch Normalization (BN)

In [8]:
# clear the name counters and set the random seed
tf.keras.backend.clear_session()
tf.random.set_seed(42)

In [9]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(300, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd",
              metrics=["accuracy"])
# model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 784)            │         3,136 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 300)            │         1,200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 271,346 (1.04 MB)

 Trainable params: 268,978 (1.03 MB)

 Non-trainable params: 2,368 (9.25 KB)

BatchNormalization adds 4 parameters per input: 2 (output scale gamma and output shift (offset) beta parameters) are affected by backpropagation, and 2 are not: mean and variance - the moving averages

In [11]:
[(var.name, var.trainable) for var in model.layers[1].variables]

[('gamma', True),
 ('beta', True),
 ('moving_mean', False),
 ('moving_variance', False)]

Sometimes applying BN before the activation function works better (there's a debate on this topic). Moreover, the layer before a BatchNormalization layer does not need to have bias terms, since the BatchNormalization layer some as well, it would be a waste of parameters, so you can set use_bias=False when creating those layers:

In [13]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(300, kernel_initializer="he_normal", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(100, kernel_initializer="he_normal", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd",
              metrics=["accuracy"])
# model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

### Gradient Clipping